In [19]:
!pip install -q ultralytics


In [20]:
import random
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import torch
import shutil

In [21]:
SEED = 42
random.seed(SEED)

SRC_IMG = Path('/kaggle/input/face-detection-dataset/images/train')
SRC_LBL = Path('/kaggle/input/face-detection-dataset/labels/train')

DST_IMG = Path('/kaggle/working/images/test')
DST_LBL = Path('/kaggle/working/labels/test')

DST_IMG.mkdir(parents=True, exist_ok=True)
DST_LBL.mkdir(parents=True, exist_ok=True)

images = list(SRC_IMG.glob('*.jpg'))
sample_imgs = random.sample(images, 300)

for img in sample_imgs:
    lbl = SRC_LBL / (img.stem + '.txt')
    if lbl.exists():
        shutil.copy(img, DST_IMG / img.name)
        shutil.copy(lbl, DST_LBL / lbl.name)

print(f"Test images: {len(list(DST_IMG.glob('*.jpg')))}")


Test images: 300


In [22]:
import yaml

data = {
    'train': '/kaggle/input/face-detection-dataset/images/train',
    'val': '/kaggle/input/face-detection-dataset/images/val',
    'test': '/kaggle/working/images/test',
    'nc': 1,
    'names': ['faces']
}

with open('/kaggle/working/data_test.yaml', 'w') as f:
    yaml.dump(data, f, sort_keys=False)

!cat /kaggle/working/data_test.yaml


train: /kaggle/input/face-detection-dataset/images/train
val: /kaggle/input/face-detection-dataset/images/val
test: /kaggle/working/images/test
nc: 1
names:
- faces


In [23]:
!yolo detect train data=/kaggle/working/data_test.yaml model=yolo11s.pt epochs=60 imgsz=640


Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_test.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pos

In [28]:
!yolo detect val \
  model=/kaggle/working/runs/detect/train6/weights/best.pt \
  data=/kaggle/working/data_test.yaml \
  split=test


Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2556.8±435.9 MB/s, size: 184.0 KB)
val: Scanning /kaggle/working/labels/test.cache... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 5.6Mit/s 0.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 4.8it/s 3.9s0.2s
                   all        300        888       0.91      0.894      0.941       0.68
Speed: 1.2ms preprocess, 7.3ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /kaggle/working/runs/detect/val3
💡 Learn more at https://docs.ultralytics.com/modes/val


Precision = 0.91

Recall = 0.894

mAP@50 = 0.941

mAP@50–95 = 0.68